In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix


In [2]:
data = np.load("dataset_mc_ml_v1_prepared.npz")

X_train = data["X_train"]
X_val   = data["X_val"]
X_test  = data["X_test"]

y_train = data["y_train"]
y_val   = data["y_val"]
y_test  = data["y_test"]


In [3]:
def build_regime_labels(y):
    # 1 = cost-driven, 0 = congestion-driven
    return (y[:, 0] > y[:, 1]).astype(int)

r_train = build_regime_labels(y_train)
r_val   = build_regime_labels(y_val)
r_test  = build_regime_labels(y_test)


In [4]:
def balance_dataset(X, r, seed=0):
    np.random.seed(seed)
    idx0 = np.where(r == 0)[0]
    idx1 = np.where(r == 1)[0]

    n = min(len(idx0), len(idx1))

    idx0_sel = np.random.choice(idx0, n, replace=False)
    idx1_sel = np.random.choice(idx1, n, replace=False)

    idx = np.concatenate([idx0_sel, idx1_sel])
    np.random.shuffle(idx)

    return X[idx], r[idx]

X_train_bal, r_train_bal = balance_dataset(X_train, r_train)


In [5]:
Xtr = torch.tensor(X_train_bal, dtype=torch.float32)
rtr = torch.tensor(r_train_bal, dtype=torch.float32).view(-1, 1)

Xva = torch.tensor(X_val, dtype=torch.float32)
rva = torch.tensor(r_val, dtype=torch.float32).view(-1, 1)

Xte = torch.tensor(X_test, dtype=torch.float32)
rte = torch.tensor(r_test, dtype=torch.float32).view(-1, 1)


In [6]:
class RegimeNet(nn.Module):
    def __init__(self, input_dim=12, hidden_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)


In [7]:
model = RegimeNet()


In [8]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)


In [9]:
n_epochs = 500
patience = 30

best_val_loss = float("inf")
patience_counter = 0
best_state = None

for epoch in range(n_epochs):
    # TRAIN
    model.train()
    optimizer.zero_grad()

    r_pred = model(Xtr)
    train_loss = criterion(r_pred, rtr)
    train_loss.backward()
    optimizer.step()

    # VALIDATION
    model.eval()
    with torch.no_grad():
        r_val_pred = model(Xva)
        val_loss = criterion(r_val_pred, rva)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_state = model.state_dict()
    else:
        patience_counter += 1

    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch}")
        break

    if epoch % 50 == 0:
        print(f"Epoch {epoch:4d} | Train {train_loss:.4f} | Val {val_loss:.4f}")


Epoch    0 | Train 0.6975 | Val 0.6898
Early stopping at epoch 30


In [10]:
model.load_state_dict(best_state)


<All keys matched successfully>

In [11]:
model.eval()
with torch.no_grad():
    r_test_prob = model(Xte).numpy().flatten()
    r_test_pred = (r_test_prob > 0.5).astype(int)

acc = accuracy_score(r_test, r_test_pred)
auc = roc_auc_score(r_test, r_test_prob)

print("=== NN regime classification ===")
print(f"Accuracy: {acc:.3f}")
print(f"ROC-AUC:  {auc:.3f}")
print("Confusion matrix:")
print(confusion_matrix(r_test, r_test_pred))



=== NN regime classification ===
Accuracy: 0.433
ROC-AUC:  0.481
Confusion matrix:
[[ 2  1]
 [16 11]]
